# Proyecto de Machine Learning
## Predicción del precio de seguros para mascotas

Este proyecto corresponde a la segunda fase del trabajo iniciado con el Análisis Exploratorio de Datos (EDA). Su objetivo es desarrollar un modelo de Machine Learning que permita estimar el precio de nuevas cotizaciones e identificar y cuantificar las variables con mayor influencia en su cálculo.

## Notebook 02 – Modelado

Este notebook contiene el entrenamiento y la comparación inicial de los modelos de regresión candidatos, siguiendo las decisiones documentadas en el **Documento 01 – Guía de Definición del Modelo** y en el **Documento 02 – Plan de Preprocesamiento**.

**Estado del notebook:** 🟢 Dataset preprocesado disponible. Modelos entrenados, comparados y persistidos para su uso en el Notebook 03 (evaluación, Carlos).


# 1. Importación de librerías

In [1]:
import numpy as np
import pandas as pd
import joblib
import os

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor


# 2. Configuración del entorno

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
np.set_printoptions(suppress=True)

RANDOM_STATE = 42

RUTA_DATA = "../data/"
RUTA_MODELS = "../models/"
os.makedirs(RUTA_MODELS, exist_ok=True)


# 3. Carga del dataset preprocesado

## Objetivo

Cargar el conjunto de datos ya preprocesado (limpio y codificado), generado al ejecutar el **Notebook 01 – Preprocesamiento**.

## ⚠️ Nota importante

Este archivo no está en el repositorio (excluido en `.gitignore` por confidencialidad). Antes de ejecutar esta celda, cada integrante debe haber ejecutado `01_preprocessing.ipynb` una vez en su propio ordenador para generarlo en local.


In [3]:
df = pd.read_csv(RUTA_DATA + "matriz_cotizaciones_grupos_preprocesada.csv")

print(df.shape)
df.head()


(67500, 79)


,pet_age_nr,is_pure_breed,is_dangerous,capital_vet_total,capital_vet_plus,capital_rc,precio_mensual,pet_species_1,pet_species_2,pet_gender_F,pet_gender_M,pet_sterilized_N,pet_sterilized_S,sterilized_when_No_aplica,sterilized_when_S2,sterilized_when_S3,plan_name_PET_VITAL,plan_name_PET3,grupo_raza_nombre_Abisinio,grupo_raza_nombre_Affenpinscher,grupo_raza_nombre_Airedale Terrier,grupo_raza_nombre_Akita Americano (G Perro Japones),grupo_raza_nombre_Akita Inu,grupo_raza_nombre_American Staffordshire Terrier,grupo_raza_nombre_Anglo-Francais De Petite Venerie,grupo_raza_nombre_Australian Terrier,grupo_raza_nombre_Balinés,grupo_raza_nombre_Beagle,grupo_raza_nombre_Biewer Terrier,grupo_raza_nombre_Boxer,grupo_raza_nombre_Bull Terrier,grupo_raza_nombre_Bullmastiff,grupo_raza_nombre_Caniche Enano (Poodle Mini),grupo_raza_nombre_Caniche Miniatura (Poodle Toy),grupo_raza_nombre_Exotic,grupo_raza_nombre_Gato Común/European Shorthair Cat,grupo_raza_nombre_Mestizo Gigante (Más de 45 Kg),grupo_raza_nombre_Mestizo Grande (Entre 21 y 45 Kg),grupo_raza_nombre_Mestizo Mediano (Entre 11 y 20 Kg),grupo_raza_nombre_Mestizo Miniatura (Menos de 5 Kg),grupo_raza_nombre_Mestizo Pequeño (Entre 5 y 10 Kg),grupo_raza_nombre_Shar Pei,grupo_raza_nombre_Spaniel Continental Enano De Compañía (Papillon / Phaléne),pet_race_name_Abisinio,pet_race_name_Affenpinscher,pet_race_name_Airedale Terrier,pet_race_name_Akita Americano (G Perro Japones),pet_race_name_Akita Inu,pet_race_name_American Staffordshire Terrier,pet_race_name_Anglo-Francais De Petite Venerie,pet_race_name_Australian Terrier,pet_race_name_Balinés,pet_race_name_Beagle,pet_race_name_Biewer Terrier,pet_race_name_Boxer,pet_race_name_Bull Terrier,pet_race_name_Bullmastiff,pet_race_name_Caniche Enano (Poodle Mini),pet_race_name_Caniche Miniatura (Poodle Toy),pet_race_name_Exotic,pet_race_name_Gato Común/European Shorthair Cat,pet_race_name_Mestizo Gigante (Más de 45 Kg),pet_race_name_Mestizo Grande (Entre 21 y 45 Kg),pet_race_name_Mestizo Mediano (Entre 11 y 20 Kg),pet_race_name_Mestizo Miniatura (Menos de 5 Kg),pet_race_name_Mestizo Pequeño (Entre 5 y 10 Kg),pet_race_name_Shar Pei,pet_race_name_Spaniel Continental Enano De Compañía (Papillon / Phaléne),owner_postal_code_2124,owner_postal_code_2210,owner_postal_code_28001,owner_postal_code_28220,owner_postal_code_50001,owner_municipality_Albacete,owner_municipality_Alcalá del Júcar,owner_municipality_Madrid,owner_municipality_Majadahonda,owner_municipality_Municipio no informado,owner_municipality_Zaragoza
0,6,1,0,1250.0,1000.0,0.0,19.30,1,0,0,1,1,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
1,7,1,0,1250.0,1000.0,0.0,18.53,1,0,1,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
2,7,1,0,1250.0,1000.0,0.0,19.08,1,0,1,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
3,7,1,0,1250.0,1000.0,0.0,19.49,1,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
4,7,1,0,1250.0,1000.0,0.0,18.86,1,0,0,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0


## División Train / Test

Se separa el dataset en entrenamiento (80%) y test (20%). El conjunto de test no se utilizará hasta la evaluación final (fase de Carlos).


In [4]:
TARGET = "precio_mensual"

# Salvaguarda: si por error quedara alguna columna de leakage, se elimina aquí también
columnas_leakage = ["precio_anual"]
columnas_a_eliminar = [c for c in columnas_leakage if c in df.columns]
if columnas_a_eliminar:
    print("Aviso: se han eliminado columnas de leakage detectadas:", columnas_a_eliminar)
    df = df.drop(columns=columnas_a_eliminar)

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "| Test:", X_test.shape)


Train: (54000, 78) | Test: (13500, 78)


## Exportar Train / Test para el Notebook 03 (Evaluación)

Carlos necesita el mismo conjunto de test (y train, para revisar overfitting) para evaluar los modelos de forma coherente con este notebook. Se guardan como pickle en `src/data/` (excluido también en `.gitignore`, igual que el resto de datos).


In [5]:
X_train.to_pickle(RUTA_DATA + "X_train.pkl")
X_test.to_pickle(RUTA_DATA + "X_test.pkl")
y_train.to_pickle(RUTA_DATA + "y_train.pkl")
y_test.to_pickle(RUTA_DATA + "y_test.pkl")

print("Train/Test exportados en", RUTA_DATA)


Train/Test exportados en ../data/


# 4. Función de evaluación de modelos

## Objetivo

Función reutilizable que entrena y evalúa cada modelo mediante validación cruzada (solo sobre train), devolviendo MAE, RMSE y R².


In [6]:
def evaluar_modelo(nombre, modelo, X_train, y_train, cv=5):
    """
    Evalúa un modelo de regresión mediante validación cruzada.
    Se aplica únicamente sobre el conjunto de train.
    """
    scoring = {
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2",
    }

    resultados = cross_validate(
        modelo, X_train, y_train,
        cv=cv,
        scoring=scoring,
    )

    return {
        "Modelo": nombre,
        "MAE": -resultados["test_MAE"].mean(),
        "RMSE": -resultados["test_RMSE"].mean(),
        "R2": resultados["test_R2"].mean(),
    }


# 5. Definición de los modelos

## Objetivo

Definir en un único diccionario todos los modelos a comparar: el baseline, los modelos lineales acordados en el Documento 01 (Regresión Lineal, Ridge, Lasso, Elastic Net) y Random Forest como término de comparación.

Usar un diccionario común facilita evaluarlos todos igual y guardarlos con nombres consistentes, que son los que utilizará Carlos en el Notebook 03.


In [7]:
modelos = {
    "Baseline (media)": DummyRegressor(strategy="mean"),
    "Regresión Lineal": LinearRegression(),
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Lasso": Lasso(random_state=RANDOM_STATE),
    "Elastic Net": ElasticNet(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE),
}


## Validación cruzada (comparación inicial)

Se evalúa cada modelo con validación cruzada sobre train, antes de entrenarlo de forma definitiva.


In [8]:
resultados_cv = [
    evaluar_modelo(nombre, modelo, X_train, y_train)
    for nombre, modelo in modelos.items()
]

tabla_comparativa = pd.DataFrame(resultados_cv).sort_values("RMSE").reset_index(drop=True)
tabla_comparativa


,Modelo,MAE,RMSE,R2
0,Random Forest,0.273011,0.535897,0.997409
1,Ridge,1.735142,2.321579,0.951464
2,Regresión Lineal,1.735341,2.321664,0.951461
3,Elastic Net,5.531146,7.330219,0.516203
4,Lasso,5.697438,7.585979,0.481851
5,Baseline (media),8.343754,10.539121,-0.000073


## Entrenamiento final y persistencia

Nombres de archivo usados: `baseline.pkl`, `linear_regression.pkl`, `ridge.pkl`, `lasso.pkl`, `elastic_net.pkl`, `random_forest.pkl` 


In [9]:
nombres_archivo = {
    "Baseline (media)": "baseline.pkl",
    "Regresión Lineal": "linear_regression.pkl",
    "Ridge": "ridge.pkl",
    "Lasso": "lasso.pkl",
    "Elastic Net": "elastic_net.pkl",
    "Random Forest": "random_forest.pkl",
}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    ruta_archivo = RUTA_MODELS + nombres_archivo[nombre]
    joblib.dump(modelo, ruta_archivo)
    print(f"Modelo '{nombre}' entrenado y guardado en {ruta_archivo}")


Modelo 'Baseline (media)' entrenado y guardado en ../models/baseline.pkl
Modelo 'Regresión Lineal' entrenado y guardado en ../models/linear_regression.pkl
Modelo 'Ridge' entrenado y guardado en ../models/ridge.pkl
Modelo 'Lasso' entrenado y guardado en ../models/lasso.pkl
Modelo 'Elastic Net' entrenado y guardado en ../models/elastic_net.pkl
Modelo 'Random Forest' entrenado y guardado en ../models/random_forest.pkl


In [11]:
# --- Validación adicional del resultado de Random Forest ---

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Confirmar que el rendimiento se mantiene sobre el conjunto de TEST (no solo en validación cruzada sobre train)
modelo_rf_final = modelos["Random Forest"]  # ya entrenado con todo el train en la celda anterior
y_pred_test = modelo_rf_final.predict(X_test)

mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = mean_squared_error(y_test, y_pred_test) ** 0.5
r2_test = r2_score(y_test, y_pred_test)

print("Random Forest sobre TEST:")
print(f"  MAE:  {mae_test:.4f}")
print(f"  RMSE: {rmse_test:.4f}")
print(f"  R2:   {r2_test:.4f}")

# 2. Repetir el entrenamiento usando SOLO variables básicas de mascota y póliza
prefijos_basicos = (
    "plan_", "capital_vet_total", "capital_vet_plus", "capital_rc",
    "pet_species", "pet_age_nr", "pet_sterilized", "grupo_raza",
    "is_pure_breed", "is_dangerous", "pet_gender",
)

columnas_basicas = [c for c in X_train.columns if c.startswith(prefijos_basicos)]
print(f"\nColumnas básicas usadas ({len(columnas_basicas)}):", columnas_basicas)

rf_basico = RandomForestRegressor(random_state=RANDOM_STATE)
rf_basico.fit(X_train[columnas_basicas], y_train)
y_pred_basico = rf_basico.predict(X_test[columnas_basicas])

r2_basico = r2_score(y_test, y_pred_basico)
print(f"\nR2 en TEST usando solo variables básicas: {r2_basico:.4f}")

Random Forest sobre TEST:
  MAE:  0.2305
  RMSE: 0.4764
  R2:   0.9980

Columnas básicas usadas (39): ['pet_age_nr', 'is_pure_breed', 'is_dangerous', 'capital_vet_total', 'capital_vet_plus', 'capital_rc', 'pet_species_1', 'pet_species_2', 'pet_gender_F', 'pet_gender_M', 'pet_sterilized_N', 'pet_sterilized_S', 'plan_name_PET_VITAL', 'plan_name_PET3', 'grupo_raza_nombre_Abisinio', 'grupo_raza_nombre_Affenpinscher', 'grupo_raza_nombre_Airedale Terrier', 'grupo_raza_nombre_Akita Americano (G Perro Japones)', 'grupo_raza_nombre_Akita Inu', 'grupo_raza_nombre_American Staffordshire Terrier', 'grupo_raza_nombre_Anglo-Francais De Petite Venerie', 'grupo_raza_nombre_Australian Terrier', 'grupo_raza_nombre_Balinés', 'grupo_raza_nombre_Beagle', 'grupo_raza_nombre_Biewer Terrier', 'grupo_raza_nombre_Boxer', 'grupo_raza_nombre_Bull Terrier', 'grupo_raza_nombre_Bullmastiff', 'grupo_raza_nombre_Caniche Enano (Poodle Mini)', 'grupo_raza_nombre_Caniche Miniatura (Poodle Toy)', 'grupo_raza_nombre_Exotic

### Conclusiones
Random Forest obtiene el mejor rendimiento con diferencia (R²=0.997, RMSE=0.54), seguido de cerca por Ridge y Regresión Lineal (R²≈0.95). Lasso y Elastic Net obtuvieron un rendimiento notablemente inferior, probablemente por una regularización demasiado agresiva con los hiperparámetros por defecto (alpha=1.0), dado el elevado número de variables tras el encoding (79 columnas). Todos los modelos superan claramente al baseline, confirmando que las variables del dataset tienen poder predictivo real sobre el precio.

### Decisión
Se seleccionan Random Forest y Ridge para la fase de optimización de hiperparámetros: Random Forest por su rendimiento superior, y Ridge por ofrecer un resultado casi idéntico a Regresión Lineal pero con mayor robustez y mejor interpretabilidad mediante coeficientes. Se valorará también reoptimizar Lasso/Elastic Net con valores de alpha más bajos.


### Validación del resultado de Random Forest

Se comprobaron tres aspectos para descartar un resultado artificial:

1. **Rendimiento sobre el conjunto de test** (nunca visto durante el entrenamiento): 
   MAE=0.23, RMSE=0.48, R²=0.998 — prácticamente idéntico al resultado de validación 
   cruzada sobre train (R²=0.997), lo que confirma que el modelo generaliza correctamente 
   y no hay sobreajuste.

2. **Rendimiento usando solo variables básicas** de mascota y póliza (plan, capitales, 
   especie, edad, esterilización, raza, peligrosidad, sexo — sin código postal ni otras 
   variables más detalladas): R²=0.9223. Esto demuestra que el precio del seguro es, por 
   su propia naturaleza, altamente predecible a partir de un conjunto reducido de variables, 
   coherente con un cálculo de tipo actuarial/tarifario propio del sector seguros.

3. **Ausencia de registros duplicados o casi-duplicados**: se verificó sobre el dataset 
   original que no existen filas idénticas (excluyendo identificadores y fecha) que pudieran 
   facilitar artificialmente la predicción.

**Conclusión:** el alto R² de Random Forest (0.997-0.998)se debe que el precio del seguro sigue un patrón muy 
determinista a partir de las variables disponibles.

# 6. Cierre

Con el dataset preprocesado ya integrado, este notebook entrena, compara y persiste el modelo baseline, los modelos lineales y Random Forest, dejando todo preparado (modelos guardados + train/test exportado) para que el Notebook 03 de evaluación pueda ejecutarse directamente. El siguiente paso será seleccionar los 2-3 mejores modelos según la tabla comparativa y pasar a la fase de optimización de hiperparámetros (GridSearchCV / RandomizedSearchCV con validación cruzada).
